# 03 — ML Models (LightGBM + XGBoost) on the Unified Feature Table

Both ML models share the **same** feature columns, the **same** target convention (predict `weekly_units` of week W using only past info), and the **same** two-stage architecture:

- **Stage 1** — occurrence classifier `P(weekly_units > 0)`, with class-imbalance correction.
- **Stage 2** — magnitude regressor on `log1p(weekly_units)`, trained on positive rows only.
- **Final prediction** — `y_pred = P * E[units|positive]` (a **soft** combination, no hard gate).

Why soft instead of hard gate? With ~78% of validation rows being true zeros, a "predict all zeros" baseline already achieves WAPE = 1.0; any hard threshold that fires non-trivial predictions tends to over-shoot in WAPE because the magnitude regressor (trained only on positives) cannot output zero. The soft combination `P * mag` smoothly down-weights low-probability pairs and produces a **non-zero expected demand** for every (facility, product) row — which is exactly what the shopping list needs.

**Inputs:** `unified_features.parquet`.
**Outputs:** `data/lgbm_predictions.csv`, `data/xgb_predictions.csv`.


## Step 0 — Imports and split

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

DATA_DIR = Path.cwd() / 'data'
PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']

uni = pd.read_parquet(DATA_DIR / 'unified_features.parquet')
print('Unified features shape:', uni.shape)
print('Per-nest x split row counts:')
print(uni.groupby(['NEST_NAME', 'split']).size().unstack(fill_value=0))


## Step 1 — Define the unified feature column list

In [ ]:
FEATURES = [
    # calendar
    'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
    # lags
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
    # rollings
    'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
    'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
    'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
    # intermittent
    'weeks_since_last_demand', 'last_nonzero_units',
    'historical_nonzero_rate', 'cumulative_active_weeks',
    # expanding history
    'facility_hist_mean', 'facility_hist_nz_rate',
    'product_hist_mean',  'product_hist_nz_rate',
    'fp_hist_mean',       'fp_hist_nz_rate',
    # pair-level numerics
    'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
    # encodings
    'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
]
print(f'{len(FEATURES)} unified features.')

train = uni[uni['split'] == 'train'].dropna(subset=['lag_1']).reset_index(drop=True)
val   = uni[uni['split'] == 'valid'].reset_index(drop=True)
test  = uni[uni['split'] == 'test' ].reset_index(drop=True)

X_tr = train[FEATURES].fillna(-999); y_tr = train['weekly_units'].astype(float)
X_va = val[FEATURES].fillna(-999);   y_va = val['weekly_units'].astype(float)
X_te = test[FEATURES].fillna(-999);  y_te = test['weekly_units'].astype(float)

occ_tr = (y_tr > 0).astype(int)
occ_va = (y_va > 0).astype(int)
n_pos_tr = int(occ_tr.sum()); n_neg_tr = len(occ_tr) - n_pos_tr
spw = n_neg_tr / max(n_pos_tr, 1)
pos_mask_tr = (y_tr > 0).to_numpy()
pos_mask_va = (y_va > 0).to_numpy()
print(f'Rows  train={len(train):,}  valid={len(val):,}  test={len(test):,}')
print(f'Train occurrence balance: pos={n_pos_tr:,}  neg={n_neg_tr:,}  scale_pos_weight={spw:.2f}')


## Step 2 — Helper: soft two-stage prediction + scoring

`y_pred = prob * magnitude` (clipped at 0). This is the **expected** value of a Bernoulli mixture: occurrence probability times conditional mean magnitude.

In [ ]:
def soft_predict(clf, reg, X):
    prob = clf.predict_proba(X)[:, 1]
    mag  = np.expm1(reg.predict(X)).clip(min=0)
    return np.clip(prob * mag, 0, None), prob, mag


def wape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.sum(np.abs(p - y)) / max(np.sum(np.abs(y)), 1e-9))


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(p, dtype=float) - np.asarray(y, dtype=float))))


## Step 3 — LightGBM — Stage 1 (occurrence classifier)

In [ ]:
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, random_state=42, verbosity=-1,
)
lgbm_clf.fit(X_tr, occ_tr, eval_set=[(X_va, occ_va)],
             eval_metric='auc', callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print(f'LGBM Stage-1 best iteration: {lgbm_clf.best_iteration_}')


## Step 4 — LightGBM — Stage 2 (magnitude regressor on positive rows)

In [ ]:
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=-1,
)
lgbm_reg.fit(X_tr[pos_mask_tr], np.log1p(y_tr[pos_mask_tr]),
             eval_set=[(X_va[pos_mask_va], np.log1p(y_va[pos_mask_va]))],
             eval_metric='rmse',
             callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print(f'LGBM Stage-2 best iteration: {lgbm_reg.best_iteration_}')

# Quick valid sanity check on the soft prediction.
val_pred, val_prob, val_mag = soft_predict(lgbm_clf, lgbm_reg, X_va)
print(f'LGBM valid soft prediction: MAE={mae(y_va, val_pred):.4f}  WAPE={wape(y_va, val_pred):.4f}')


## Step 5 — LightGBM — test prediction + per-nest snapshot

In [ ]:
lgbm_test_pred, lgbm_test_prob, lgbm_test_mag = soft_predict(lgbm_clf, lgbm_reg, X_te)
out_lgbm = test[PAIR_COLS + ['week_start', 'demand_class']].copy()
out_lgbm['y_actual'] = y_te.values
out_lgbm['y_pred']   = lgbm_test_pred
out_lgbm['s1_prob']  = lgbm_test_prob
out_lgbm['s2_mag']   = lgbm_test_mag
out_lgbm['method']   = 'LGBM_two_stage_soft'
out_lgbm.to_csv(DATA_DIR / 'lgbm_predictions.csv', index=False)

snap = (out_lgbm.groupby('NEST_NAME')
                 .apply(lambda g: pd.Series({'n': len(g),
                                             'MAE':  mae(g['y_actual'], g['y_pred']),
                                             'WAPE': wape(g['y_actual'], g['y_pred'])}))
                 .reset_index())
print('LGBM per-nest test snapshot (soft prediction):')
print(snap.to_string(index=False))
print('\nSaved lgbm_predictions.csv')


## Step 6 — XGBoost — Stage 1 (occurrence classifier)

Same architecture, different gradient-boosting library.

In [ ]:
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='aucpr',
    n_estimators=600, learning_rate=0.05, max_depth=5, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.8, gamma=0.5, reg_alpha=0.1, reg_lambda=5.0,
    scale_pos_weight=spw, random_state=42, n_jobs=-1,
    early_stopping_rounds=30, verbosity=0,
)
xgb_clf.fit(X_tr, occ_tr, eval_set=[(X_va, occ_va)], verbose=False)
print(f'XGB Stage-1 best iteration: {xgb_clf.best_iteration}')


## Step 7 — XGBoost — Stage 2 (magnitude regressor)

In [ ]:
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror', eval_metric='rmse',
    n_estimators=600, learning_rate=0.05, max_depth=4, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, gamma=0.5, reg_alpha=0.1, reg_lambda=3.0,
    random_state=42, n_jobs=-1, early_stopping_rounds=30, verbosity=0,
)
xgb_reg.fit(X_tr[pos_mask_tr], np.log1p(y_tr[pos_mask_tr]),
            eval_set=[(X_va[pos_mask_va], np.log1p(y_va[pos_mask_va]))],
            verbose=False)
print(f'XGB Stage-2 best iteration: {xgb_reg.best_iteration}')

val_pred_xgb, _, _ = soft_predict(xgb_clf, xgb_reg, X_va)
print(f'XGB valid soft prediction: MAE={mae(y_va, val_pred_xgb):.4f}  WAPE={wape(y_va, val_pred_xgb):.4f}')


## Step 8 — XGBoost — test prediction + per-nest snapshot

In [ ]:
xgb_test_pred, xgb_test_prob, xgb_test_mag = soft_predict(xgb_clf, xgb_reg, X_te)
out_xgb = test[PAIR_COLS + ['week_start', 'demand_class']].copy()
out_xgb['y_actual'] = y_te.values
out_xgb['y_pred']   = xgb_test_pred
out_xgb['s1_prob']  = xgb_test_prob
out_xgb['s2_mag']   = xgb_test_mag
out_xgb['method']   = 'XGB_two_stage_soft'
out_xgb.to_csv(DATA_DIR / 'xgb_predictions.csv', index=False)

snap = (out_xgb.groupby('NEST_NAME')
                .apply(lambda g: pd.Series({'n': len(g),
                                            'MAE':  mae(g['y_actual'], g['y_pred']),
                                            'WAPE': wape(g['y_actual'], g['y_pred'])}))
                .reset_index())
print('XGB per-nest test snapshot (soft prediction):')
print(snap.to_string(index=False))
print('\nSaved xgb_predictions.csv')
